# Fase 1 · M04d: Corrección de Vía de Acceso

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 1 — Transformación |
| **Módulo** | M04d — Corrección Vía Acceso |

---

## 🎯 Qué hace

Corrige y estandariza los valores de la variable via_acceso en el dataset de alumnos.

## 📋 Requisitos

- `data/02_processed/df_alumno.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/02_processed/df_alumno.parquet` | Dataset con vía de acceso corregida (sobreescribe) |

## 🔄 Flujo

```
df_alumno.parquet
    ↓ Detección de vías incorrectas/inconsistentes
    ↓ Estandarización
    → df_alumno.parquet (corregido)
```

## ➡️ Siguiente

`f1_m05_dashboard.ipynb` — dashboard resumen de Fase 1


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# --- Detectar entorno ---
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
from src.config import RUTA_PROCESSED, info_entorno

# --- Info entorno ---
info_entorno()
print(f'\n📂 Processed: {RUTA_PROCESSED}')

✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ============================

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS
# ============================================================================

print('=' * 60)
print('CARGANDO DATOS')
print('=' * 60)

path_alumno = RUTA_PROCESSED / 'df_alumno.parquet'
if not path_alumno.exists():
    raise FileNotFoundError(
        f'No encontrado: {path_alumno}\n'
        f'Ejecuta primero f1_m04c_correccion_notas.ipynb'
    )

df = pd.read_parquet(path_alumno)
print(f'✅ df_alumno: {len(df):,} registros × {len(df.columns)} columnas')

CARGANDO DATOS


✅ df_alumno: 109,568 registros × 37 columnas


In [3]:
# ============================================================================
# CELDA 3: DIAGNÓSTICO — DOS VÍAS DE ACCESO
# ============================================================================
# df_alumno tiene dos columnas que representan la vía de acceso:
#   - 'nombre' (campo 7): viene de expedientes. 20 categorías, 0 NaN.
#     Mezcla vía de acceso real (General, Selectividad, FP) con cupo
#     (Minusválidos, Deportistas) y tipo de matrícula (Adaptación).
#   - 'via_acceso' (campo 34): viene de preinscripción. 11 categorías,
#     12.6% NaN. Más limpia pero con duplicados por espaciado.
# ============================================================================

print('=' * 60)
print('DIAGNÓSTICO: DOS VÍAS DE ACCESO')
print('=' * 60)

n_total = len(df)

# --- Campo 'nombre' (expedientes) ---
na_nombre = df['nombre'].isna().sum()
print(f'\n📊 Campo "nombre" (de expedientes):')
print(f'   Categorías: {df["nombre"].nunique()}')
print(f'   NaN: {na_nombre} ({na_nombre/n_total*100:.1f}%)')
print(f'\n   Valores:')
for val, cnt in df['nombre'].value_counts().items():
    print(f'     {val:<55s} {cnt:>7,} ({cnt/n_total*100:.1f}%)')

# --- Campo 'via_acceso' (preinscripción) ---
na_via = df['via_acceso'].isna().sum()
print(f'\n📊 Campo "via_acceso" (de preinscripción):')
print(f'   Categorías: {df["via_acceso"].nunique()}')
print(f'   NaN: {na_via:,} ({na_via/n_total*100:.1f}%)')
print(f'\n   Valores:')
for val, cnt in df['via_acceso'].value_counts().items():
    print(f'     {val:<55s} {cnt:>7,} ({cnt/n_total*100:.1f}%)')

# Guardar métricas
antes_via_nan = na_via
antes_via_cats = df['via_acceso'].nunique()

DIAGNÓSTICO: DOS VÍAS DE ACCESO

📊 Campo "nombre" (de expedientes):
   Categorías: 20
   NaN: 0 (0.0%)

   Valores:
     General                                                  72,725 (66.4%)
     Selectividad                                             21,812 (19.9%)
     Adaptación a Grado                                        5,598 (5.1%)
     Traslado ( a segundo o mas )                              2,511 (2.3%)
     Majores de 25 años                                        1,972 (1.8%)
     Titulados Universitarios                                  1,201 (1.1%)
      Acceso a Grado con título previo                           822 (0.8%)
     Segunda titulación                                          669 (0.6%)
     Minusválidos                                                523 (0.5%)
     Deportistas de élite                                        484 (0.4%)
     Formacion Profesional/Modulo 3/Ciclos formativos            290 (0.3%)
     Traslado de estudios extranjeros         

In [4]:
# ============================================================================
# CELDA 4: DIAGNÓSTICO — ¿QUÉ 'nombre' TIENEN LOS SIN via_acceso?
# ============================================================================

print('=' * 60)
print('LOS 13.835 SIN via_acceso — ¿QUÉ SON?')
print('=' * 60)

sin_via = df[df['via_acceso'].isna()]
print(f'\nRegistros sin via_acceso: {len(sin_via):,}')
print(f'\nPor campo "nombre":')
for val, cnt in sin_via['nombre'].value_counts().items():
    print(f'  {val:<55s} {cnt:>6,}')

print(f'\n📝 La mayoría son Adaptación a Grado (5.502) y Traslado (1.956):')
print(f'   alumnos que no pasaron por preinscripción ordinaria.')
print(f'   No tienen via_acceso de preinscripción porque nunca la hicieron.')

LOS 13.835 SIN via_acceso — ¿QUÉ SON?

Registros sin via_acceso: 13,835

Por campo "nombre":
  Adaptación a Grado                                       5,502
  Selectividad                                             2,013
  General                                                  1,987
  Traslado ( a segundo o mas )                             1,956
   Acceso a Grado con título previo                          787
  Segunda titulación                                         657
  Traslado de estudios extranjeros                           268
  Formacion Profesional/Modulo 3/Ciclos formativos           245
  Cambio de plan de estudios de grado                        148
  Majores de 25 años                                          84
  Titulados Universitarios                                    79
  Minusválidos                                                25
  Deportistas de élite                                        20
  Mayores de 45 años                                          

In [5]:
# ============================================================================
# CELDA 5: CORRECCIÓN 1 — LIMPIAR DUPLICADOS EN via_acceso
# ============================================================================
# Hay 3 pares de duplicados por espaciado:
#   "mayores 25 años" vs "mayores 25años"
#   "mayores 40 años" vs "mayores 40años"
#   "mayores 45 años" vs "mayores 45años"
# ============================================================================

print('=' * 60)
print('CORRECCIÓN 1: LIMPIAR DUPLICADOS EN via_acceso')
print('=' * 60)

cats_antes = df['via_acceso'].nunique()

# Insertar espacio entre dígito y "años" donde falte
df['via_acceso'] = df['via_acceso'].str.replace(
    r'(\d)(años)', r'\1 \2', regex=True
)

cats_despues = df['via_acceso'].nunique()
print(f'\nCategorías: {cats_antes} → {cats_despues} (fusionados {cats_antes - cats_despues} duplicados)')
print(f'\nValores tras limpieza:')
for val, cnt in df['via_acceso'].value_counts().items():
    print(f'  {val:<55s} {cnt:>7,}')

CORRECCIÓN 1: LIMPIAR DUPLICADOS EN via_acceso

Categorías: 11 → 8 (fusionados 3 duplicados)

Valores tras limpieza:
  Pruebas acceso Bachiller Logse                           77,267
  Ciclo Formativo de Grado sup. o equivalente              13,673
  Pruebas acceso mayores 25 años                            1,974
  Titulados Universitarios                                  1,285
  Extranjeros CEE                                             959
  Extranjeros no CEE                                          284
  Pruebas acceso mayores 40 años                              162
  Pruebas acceso mayores 45 años                              129


In [6]:
# ============================================================================
# CELDA 6: CORRECCIÓN 2 — RELLENAR NaN CON MAPEO DESDE 'nombre'
# ============================================================================
# Para los 13.835 registros sin via_acceso, mapeamos desde 'nombre'.
# Hay dos tipos de mapeo:
#   a) Mapeo directo: General → Bachiller Logse, FP → Ciclo Formativo, etc.
#   b) Categoría propia: Adaptación a Grado, Traslado, etc. son formas
#      legítimas de acceso que no encajan en las categorías de preinscripción.
# ============================================================================

print('=' * 60)
print('CORRECCIÓN 2: RELLENAR NaN DESDE "nombre"')
print('=' * 60)

MAPEO_NOMBRE_A_VIA = {
    # --- Mapeo directo a categorías de preinscripción ---
    'General':                                      'Pruebas acceso Bachiller Logse',
    'Selectividad':                                 'Pruebas acceso Bachiller Logse',
    'Formacion Profesional/Modulo 3/Ciclos formativos': 'Ciclo Formativo de Grado sup. o equivalente',
    'Majores de 25 años':                           'Pruebas acceso mayores 25 años',
    'Mayores de 40 años':                           'Pruebas acceso mayores 40 años',
    'Mayores de 45 años':                           'Pruebas acceso mayores 45 años',
    'Titulados Universitarios':                     'Titulados Universitarios',
    'Minusválidos - Mayores de 25 años':            'Pruebas acceso mayores 25 años',
    'Minusválidos - Selectividad':                  'Pruebas acceso Bachiller Logse',
    'Deportistas de élite - Titulados Universitarios': 'Titulados Universitarios',
    'Euruji':                                       'Extranjeros CEE',
    'Traslado de estudios extranjeros':             'Extranjeros no CEE',
    ' Acceso a Grado con título previo':            'Titulados Universitarios',
    'Segunda titulación':                           'Titulados Universitarios',
    'Intercambio entrante':                         'Extranjeros CEE',
    # --- Categorías propias (no encajan en preinscripción) ---
    'Adaptación a Grado':                           'Adaptación a Grado',
    'Traslado ( a segundo o mas )':                 'Traslado',
    'Cambio de plan de estudios de grado':          'Cambio de plan',
    'Deportistas de élite':                         'Deportistas de élite',
    'Minusválidos':                                 'Minusválidos',
}

mask_nan = df['via_acceso'].isna()
n_nan_antes = mask_nan.sum()

# Aplicar mapeo
mapped = df.loc[mask_nan, 'nombre'].map(MAPEO_NOMBRE_A_VIA)
df.loc[mask_nan, 'via_acceso'] = mapped

n_nan_despues = df['via_acceso'].isna().sum()
n_rellenados = n_nan_antes - n_nan_despues

print(f'\nNaN antes:  {n_nan_antes:,}')
print(f'Rellenados: {n_rellenados:,}')
print(f'NaN después: {n_nan_despues:,}')

if n_nan_despues > 0:
    print(f'\n⚠️ Quedan {n_nan_despues} sin mapear:')
    print(df.loc[df['via_acceso'].isna(), 'nombre'].value_counts().to_string())
else:
    print(f'\n✅ Todos los registros tienen via_acceso')

CORRECCIÓN 2: RELLENAR NaN DESDE "nombre"

NaN antes:  13,835
Rellenados: 13,835
NaN después: 0

✅ Todos los registros tienen via_acceso


In [7]:
# ============================================================================
# CELDA 7: CORRECCIÓN 3 — RENOMBRAR 'nombre' → 'via_acceso_exp'
# ============================================================================
# El campo 'nombre' realmente es la vía de acceso que consta en expedientes.
# Lo renombramos para evitar confusión con un nombre de persona.
# Se mantiene porque tiene información complementaria (cupo, tipo matrícula)
# que no está en 'via_acceso'.
# ============================================================================

print('=' * 60)
print('CORRECCIÓN 3: RENOMBRAR nombre → via_acceso_exp')
print('=' * 60)

df = df.rename(columns={'nombre': 'via_acceso_exp'})
print(f'\n✅ Renombrado: nombre → via_acceso_exp')
print(f'   Columnas actuales: {len(df.columns)}')

CORRECCIÓN 3: RENOMBRAR nombre → via_acceso_exp

✅ Renombrado: nombre → via_acceso_exp
   Columnas actuales: 37


In [8]:
# ============================================================================
# CELDA 8: VALIDACIÓN
# ============================================================================

print('=' * 60)
print('VALIDACIÓN')
print('=' * 60)

# --- via_acceso ---
assert df['via_acceso'].isna().sum() == 0, 'ERROR: via_acceso tiene NaN'
print(f'\n✅ via_acceso: 0 NaN')
print(f'   Categorías: {df["via_acceso"].nunique()}')

# --- via_acceso_exp ---
assert df['via_acceso_exp'].isna().sum() == 0, 'ERROR: via_acceso_exp tiene NaN'
assert 'nombre' not in df.columns, 'ERROR: columna nombre aún existe'
print(f'✅ via_acceso_exp: 0 NaN (renombrado desde nombre)')

# --- Integridad ---
df_orig = pd.read_parquet(path_alumno)
assert len(df) == len(df_orig), f'ERROR: filas cambiaron ({len(df_orig)} → {len(df)})'
assert len(df.columns) == len(df_orig.columns), 'ERROR: nº columnas cambió'

print(f'✅ Registros: {len(df):,} (sin cambios)')
print(f'✅ Columnas: {len(df.columns)} (sin cambios, solo renombre)')
print(f'✅ Validación superada')

VALIDACIÓN

✅ via_acceso: 0 NaN
   Categorías: 13
✅ via_acceso_exp: 0 NaN (renombrado desde nombre)
✅ Registros: 109,568 (sin cambios)
✅ Columnas: 37 (sin cambios, solo renombre)
✅ Validación superada


In [9]:
# ============================================================================
# CELDA 9: COMPARACIÓN ANTES vs DESPUÉS
# ============================================================================

print('=' * 60)
print('COMPARACIÓN ANTES vs DESPUÉS')
print('=' * 60)

print(f'\n{"MÉTRICA":<35s} {"ANTES":>10s} {"DESPUÉS":>10s}')
print('-' * 57)
print(f'{"via_acceso NaN":<35s} {antes_via_nan:>10,} {df["via_acceso"].isna().sum():>10,}')
print(f'{"via_acceso categorías":<35s} {antes_via_cats:>10} {df["via_acceso"].nunique():>10}')
print(f'{"Duplicados espaciado":<35s} {"3 pares":>10s} {"0":>10s}')
print(f'{"Campo nombre":<35s} {"nombre":>10s} {"via_acc_exp":>10s}')

print(f'\n📊 via_acceso final ({df["via_acceso"].nunique()} categorías):')
for val, cnt in df['via_acceso'].value_counts().items():
    print(f'  {val:<55s} {cnt:>7,} ({cnt/len(df)*100:.1f}%)')

COMPARACIÓN ANTES vs DESPUÉS

MÉTRICA                                  ANTES    DESPUÉS
---------------------------------------------------------
via_acceso NaN                          13,835          0
via_acceso categorías                       11         13
Duplicados espaciado                   3 pares          0
Campo nombre                            nombre via_acc_exp

📊 via_acceso final (13 categorías):
  Pruebas acceso Bachiller Logse                           81,272 (74.2%)
  Ciclo Formativo de Grado sup. o equivalente              13,918 (12.7%)
  Adaptación a Grado                                        5,502 (5.0%)
  Titulados Universitarios                                  2,814 (2.6%)
  Pruebas acceso mayores 25 años                            2,059 (1.9%)
  Traslado                                                  1,956 (1.8%)
  Extranjeros CEE                                             992 (0.9%)
  Extranjeros no CEE                                          552 (0.5%

In [10]:
# ============================================================================
# CELDA 10: GUARDAR DATASET
# ============================================================================

print('=' * 60)
print('GUARDANDO DATASET')
print('=' * 60)

# --- Parquet ---
path_parquet = RUTA_PROCESSED / 'df_alumno.parquet'
df.to_parquet(path_parquet, index=False)
size_parquet = path_parquet.stat().st_size / 1024 / 1024
print(f'\n✅ {path_parquet.name}: {size_parquet:.2f} MB')

# --- CSV ---
path_csv = RUTA_PROCESSED / 'df_alumno.csv'
df.to_csv(path_csv, index=False, sep=';', decimal=',')
size_csv = path_csv.stat().st_size / 1024 / 1024
print(f'✅ {path_csv.name}: {size_csv:.2f} MB')

GUARDANDO DATASET



✅ df_alumno.parquet: 2.27 MB


✅ df_alumno.csv: 31.04 MB


In [11]:
# ============================================================================
# CELDA 11: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F1-M04d: CORRECCIÓN VÍA DE ACCESO — COMPLETADO')
print('=' * 60)

print(f'''
Correcciones aplicadas:

  1. Limpieza duplicados: 3 pares fusionados ("25años" → "25 años")
  2. Cascada nombre → via_acceso: {n_rellenados:,} NaN rellenados
     - Mapeo directo a categorías existentes (General → Bachiller Logse, etc.)
     - Categorías propias para los que no encajan (Adaptación, Traslado, etc.)
  3. Renombre: nombre → via_acceso_exp (evitar confusión)

Resultado:
  via_acceso NaN:   {antes_via_nan:,} → 0 (100% cobertura)
  Categorías:       {antes_via_cats} → {df['via_acceso'].nunique()} (limpias y sin duplicados)

Archivos actualizados:
  data/02_processed/df_alumno.parquet
  data/02_processed/df_alumno.csv

📌 Siguiente: Actualizar orquestador f1_m04_dataset_final.ipynb
              Luego Fase 3 — F3-M01 (P1, P3, P6-P9)
''')


✅ F1-M04d: CORRECCIÓN VÍA DE ACCESO — COMPLETADO

Correcciones aplicadas:

  1. Limpieza duplicados: 3 pares fusionados ("25años" → "25 años")
  2. Cascada nombre → via_acceso: 13,835 NaN rellenados
     - Mapeo directo a categorías existentes (General → Bachiller Logse, etc.)
     - Categorías propias para los que no encajan (Adaptación, Traslado, etc.)
  3. Renombre: nombre → via_acceso_exp (evitar confusión)

Resultado:
  via_acceso NaN:   13,835 → 0 (100% cobertura)
  Categorías:       11 → 13 (limpias y sin duplicados)

Archivos actualizados:
  data/02_processed/df_alumno.parquet
  data/02_processed/df_alumno.csv

📌 Siguiente: Actualizar orquestador f1_m04_dataset_final.ipynb
              Luego Fase 3 — F3-M01 (P1, P3, P6-P9)

